# Урок 3.5. Введение в агентов: LLM + инструменты

**Модуль 3 «RAG: Retrieval-Augmented Generation на практике» · Курс «LLM для Python-разработчиков»**

Всё, что мы строили до сих пор, — **конвейеры**: маршрут жёсткий (поиск → промпт → генерация → проверки), и решения принимает наш код. Сегодня переворачиваем управление: **агент** — это LLM, которая *сама* решает, что делать дальше, — вызвать инструмент, вызвать другой, поискать ещё раз с другой формулировкой или ответить. Мы соберём цикл агента на чистом Python + Ollama, без единого фреймворка, — и по дороге разберём, как function calling устроен под капотом (спойлер: никакой магии, модель просто генерирует JSON).

> ⚠️ **Формат урока** — как в 3.4: все ячейки с LLM и Qdrant — guarded (нужны `ollama pull qwen2.5:3b`, `docker compose up -d` и коллекция урока 3.2). Урок написан без живых прогонов, поэтому ожидаемое поведение агента описано как *ожидания и эксперименты* — проверьте их на своей машине, результаты у маленькой модели могут отличаться. Важно: `qwen2.5:3b` обучена работе с инструментами — с моделью без такой подготовки (например, базовой из урока 1.6) ничего не выйдет.

## 🎯 Цели урока

После этого урока вы сможете:

- объяснить, что **на самом деле** происходит при «вызове функции» моделью — и почему это безопасно ровно настолько, насколько безопасен ваш код;
- описывать инструменты **JSON Schema** так, чтобы модель выбирала их правильно (описание — это промпт!);
- написать **цикл агента**: думай → действуй → наблюдай, с лимитом шагов и обработкой ошибок;
- превратить RAG из урока 3.2 в **инструмент агента** — и дать модели шанс победить канарейку «Упал прод» переформулировкой запроса;
- решать, когда нужен агент, а когда **конвейер лучше** — по цене, латентности и предсказуемости.

## План

1. Конвейер против агента
2. Function calling под капотом
3. Инструменты «Векторики»: поиск и калькулятор
4. Один вызов инструмента вблизи
5. Цикл агента
6. Три эксперимента: простой вопрос, вычисление, канарейка
7. Дисциплина: безопасность, лимиты, бюджет

⏱ **Время:** ~75–90 минут.

## 1. Конвейер против агента

Наш RAG-конвейер (уроки 3.1–3.4) устроен как цех с одной технологической картой:

```text
КОНВЕЙЕР (маршрут фиксирован нашим кодом):
вопрос -> retrieve -> промпт -> генерация -> проверки -> ответ
          (1 поиск, 1 формулировка, 1 попытка - всегда)
```

Для 90% вопросов это идеально: дёшево (1–2 вызова LLM), предсказуемо, легко мерить (урок 3.3). Но вспомните канарейку: «Упал прод, что делать?» — лексический ретривер ищет слова «упал» и «прод», находит мусор, и конвейер **обречён**: у него нет хода «поискать иначе». Мы, люди, поступили бы просто — переформулировали бы запрос: «инциденты», «дежурный», «SEV1». Конвейер так не умеет. Агент — умеет:

```text
АГЕНТ (маршрут выбирает модель, наш код - руки):
        ┌─────────────────────────────────────────────┐
        ▼                                             │
вопрос -> LLM решает: [вызвать инструмент] ── выполняем,
             │            search_kb("упал прод")      │ результат - в диалог
             │            search_kb("инциденты")  <───┘ (и снова думать)
             │            calc("28-12")
             ▼
          [ответить] -> финальный ответ пользователю
```

Модель крутится в цикле «**думай → действуй → наблюдай**»: смотрит на диалог, решает, что ей не хватает, вызывает инструмент, видит результат — и решает снова. Наш код при этом не выбирает маршрут, но держит всё остальное: выполняет инструменты, ограничивает число шагов, валидирует аргументы.

Когда что выбирать — инженерное решение, а не мода:

| | конвейер | агент |
|---|---|---|
| маршрут | известен заранее | модель выбирает на ходу |
| цена | 1–2 вызова LLM | вызов LLM **на каждый шаг** |
| латентность | предсказуема | каждый шаг добавляет секунды |
| отладка и eval | простые (урок 3.3) | нужен трейс шагов, исходов больше |
| сильная сторона | массовые однотипные вопросы | задачи, где маршрут заранее неизвестен |

Правило по опыту: **начинайте с конвейера; агент — когда маршрут действительно вариативен.** Наш сервис 3.2 правильно устроен как конвейер, и агент его не заменит — он его *дополнит* для сложных случаев.

## 2. Function calling под капотом

Слова «модель вызывает функцию» описывают происходящее плохо. Модель **ничего не вызывает** — у неё нет ни рук, ни сети, ни интерпретатора. Происходит вот что:

1. Мы отправляем в API, вместе с сообщениями, **описания инструментов** — имя, назначение, параметры в формате JSON Schema (тот же формат, каким Pydantic описывает модели, — урок 1.8).
2. Модель, обученная этому формату, вместо текстового ответа может сгенерировать **структурный кусок**: «хочу вызвать `search_kb` с аргументами `{"query": "отпуск"}`». Это просто JSON, который она *написала* — так же, как пишет любой другой текст.
3. **Наш код** этот JSON читает, находит настоящую функцию, вызывает её и кладёт результат обратно в диалог сообщением с ролью `tool`.
4. Модель читает результат и решает: вызвать что-то ещё или ответить.

Из этого следуют две вещи, которые стоит запомнить навсегда:

- **Вся власть — у нашего кода.** Модель может «попросить» что угодно, выполняем — мы. Поэтому аргументы инструмента — это **недоверенный ввод** (модель галлюцинирует и в аргументах!), и валидировать их надо как пользовательский ввод с формы.
- **Описание инструмента — это промпт.** Модель выбирает инструмент, читая `name` и `description`. Расплывчатое описание → неверный выбор или лишние вызовы. Хорошее описание говорит, *когда* инструментом пользоваться и *как* — увидите в нашем `search_kb`.

Протокол Ollama (`POST /api/chat`) выглядит так:

```jsonc
// запрос: сообщения + инструменты
{"model": "qwen2.5:3b",
 "messages": [...],
 "tools": [{"type": "function",
            "function": {"name": "search_kb",
                         "description": "...",
                         "parameters": {/* JSON Schema */}}}]}

// ответ, если модель решила вызвать инструмент:
{"message": {"role": "assistant",
             "tool_calls": [{"function": {"name": "search_kb",
                                          "arguments": {"query": "отпуск"}}}]}}

// наш ход: кладём результат в диалог и зовём модель снова
{"role": "tool", "tool_name": "search_kb", "content": "[1] Отпуск: ..."}
```

Заметьте: в Ollama `arguments` приходят уже **разобранным объектом** (в OpenAI-совместимых API это строка с JSON, которую надо парсить самому, — деталь, о которую спотыкаются при переносе кода). И ещё: модель может запросить **несколько вызовов сразу** (`tool_calls` — список) — цикл обязан обработать все.

In [ ]:
import ast
import json
import operator
import os
import subprocess
import sys
from pathlib import Path

import httpx

sys.path.insert(0, os.getcwd())

QDRANT_URL = os.getenv("QDRANT_URL", "http://127.0.0.1:6333")
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://127.0.0.1:11434")
MODEL = os.getenv("LLM_MODEL", "qwen2.5:3b")
COLLECTION = "vectorika_rag_tfidf"
STATE_PATH = Path("service/tfidf_state.json")


def check(label, probe):
    try:
        detail = probe()
        print(f"[ OK ] {label}: {detail}")
        return True
    except Exception as error:
        print(f"[SKIP] {label}: {type(error).__name__} - зависимые ячейки пропустятся")
        return False


QDRANT_AVAILABLE = check(
    "Qdrant", lambda: "v" + httpx.get(QDRANT_URL, timeout=3).json()["version"])
OLLAMA_AVAILABLE = check(
    "Ollama и модель",
    lambda: MODEL if MODEL in [m["name"] for m in httpx.get(
        f"{OLLAMA_URL}/api/tags", timeout=3).json()["models"]] else 1 / 0)

if QDRANT_AVAILABLE:
    from qdrant_client import QdrantClient

    from service.embedder import TfidfEmbedder

    if not STATE_PATH.exists():
        subprocess.run([sys.executable, "service/index_kb.py"], check=False)
    emb = TfidfEmbedder.load(STATE_PATH)
    qdrant = QdrantClient(url=QDRANT_URL)

    def retrieve(question: str, top_k: int = 3) -> list[dict]:
        hits = qdrant.query_points(COLLECTION, query=emb.embed_query(question),
                                   limit=top_k).points
        return [{**hit.payload, "score": hit.score} for hit in hits]


def chat_with_tools(messages: list, tools: list) -> dict:
    """Один ход модели. Возвращает message: текст ИЛИ запрос инструментов."""
    response = httpx.post(f"{OLLAMA_URL}/api/chat", json={
        "model": MODEL, "messages": messages, "tools": tools, "stream": False,
        "options": {"temperature": 0.0, "num_predict": 500},
    }, timeout=300)
    response.raise_for_status()
    return response.json()["message"]


print("настройка готова")

## 3. Инструменты «Векторики»: поиск и калькулятор

Даём агенту два инструмента — на разные классы задач:

1. **`search_kb`** — наш ретривер из урока 3.2, завёрнутый в интерфейс инструмента. Это «агентный RAG»: не мы решаем, когда искать, — модель. Обратите внимание на description: в нём прямо написано «переформулируй запрос, если не нашлось» — это и есть «описание = промпт»: мы вкладываем стратегию в текст, который модель прочитает.
2. **`calc`** — арифметика. LLM считает в уме плохо (вспомните «56 дней» из 3.3!), а вопросы вида «сколько дней отпуска у меня останется?» требуют именно счёта. И здесь первая ловушка безопасности: аргумент приходит **от модели**, то есть недоверенный. Никакого `eval()` — только разбор через `ast` с белым списком операций.

Инструмент возвращает **строку**: результат уйдёт в диалог как текст сообщения с ролью `tool`, другого канала у модели нет. Ошибка — тоже строка: агент, увидев «ошибка: ...», может исправиться на следующем шаге, а упавший с исключением цикл не может ничего.

In [ ]:
# --- инструмент 1: поиск по базе знаний (ретривер урока 3.2) -----------------
def tool_search_kb(query: str) -> str:
    if not QDRANT_AVAILABLE:
        return "ошибка: поиск недоступен"
    found = retrieve(str(query), top_k=3)
    if not found or found[0]["score"] < 0.05:      # совсем мусор - честно скажем
        return "ничего похожего не найдено, попробуй другую формулировку"
    return "\n\n".join(
        f"[{c['doc_id']}] {c['title']} (score {c['score']:.2f}):\n{c['text']}"
        for c in found)


# --- инструмент 2: калькулятор БЕЗ eval ---------------------------------------
SAFE_OPS = {ast.Add: operator.add, ast.Sub: operator.sub,
            ast.Mult: operator.mul, ast.Div: operator.truediv,
            ast.USub: operator.neg}


def tool_calc(expression: str) -> str:
    """Арифметика через ast: числа, + - * /, скобки. Всё прочее - отказ.
    eval() здесь был бы дырой: аргумент генерирует модель, это недоверенный ввод."""

    def evaluate(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in SAFE_OPS:
            return SAFE_OPS[type(node.op)](evaluate(node.left), evaluate(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in SAFE_OPS:
            return SAFE_OPS[type(node.op)](evaluate(node.operand))
        raise ValueError("допустимы только числа, + - * / и скобки")

    try:
        result = evaluate(ast.parse(str(expression), mode="eval").body)
        return str(round(result, 6))
    except Exception as error:
        return f"ошибка вычисления: {error}"


# --- описания для модели: JSON Schema (как parameters у Pydantic) --------------
TOOLS = [
    {"type": "function", "function": {
        "name": "search_kb",
        "description": (
            "Поиск по внутренней базе знаний компании «Векторика» (отпуска, "
            "больничные, дежурства, деплой, код-ревью и т.п.). Возвращает три "
            "наиболее похожих фрагмента документов. Используй для ЛЮБОГО вопроса "
            "о правилах компании. Если результат не отвечает на вопрос — вызови "
            "поиск ещё раз с другой формулировкой (синонимы, официальные термины: "
            "например, вместо «упал прод» — «инцидент», «дежурный»)."),
        "parameters": {"type": "object",
                       "properties": {"query": {
                           "type": "string",
                           "description": "поисковый запрос по-русски"}},
                       "required": ["query"]},
    }},
    {"type": "function", "function": {
        "name": "calc",
        "description": ("Калькулятор: числа, скобки и операции + - * /. "
                        "Используй для любой арифметики вместо счёта в уме."),
        "parameters": {"type": "object",
                       "properties": {"expression": {
                           "type": "string",
                           "description": "выражение, например (28-12)*2"}},
                       "required": ["expression"]},
    }},
]

TOOL_HANDLERS = {"search_kb": tool_search_kb, "calc": tool_calc}

# калькулятор можно проверить прямо сейчас, без всякой LLM:
print(tool_calc("28 - 12"))                     # ожидаем 16
print(tool_calc("(28 - 12) / 2"))               # ожидаем 8.0
print(tool_calc("__import__('os').getcwd()"))   # ожидаем отказ, а не выполнение!

Последняя строка ячейки — мини-тест безопасности: строка `__import__('os').getcwd()` — валидный Python, и `eval()` его **выполнил бы**. Наш разборщик вернёт «ошибка вычисления: допустимы только числа...» — код, пришедший от модели, не исполняется никогда, исполняется только арифметика из белого списка.

## 4. Один вызов инструмента вблизи

Прежде чем строить цикл, посмотрим на один ход в замедленной съёмке: отправим вопрос с инструментами и напечатаем, что модель *сгенерировала*. Ответа пользователю здесь ещё нет — есть **намерение**, записанное как JSON:

In [ ]:
if OLLAMA_AVAILABLE:
    messages = [
        {"role": "system", "content": "Ты — ассистент компании «Векторика». "
         "Пользуйся инструментами, не отвечай из памяти."},
        {"role": "user", "content": "Сколько дней отпуска положено в году?"},
    ]
    message = chat_with_tools(messages, TOOLS)
    print("модель вернула message:")
    print(json.dumps(message, ensure_ascii=False, indent=2))
    # ожидаем: "tool_calls": [{"function": {"name": "search_kb",
    #                                       "arguments": {"query": "..."}}}]
    # и пустой/отсутствующий content - модель пока не отвечает, а действует
else:
    print("[пропущено: Ollama недоступна]")

Что здесь стоит разглядеть (запустив у себя):

- `tool_calls` — **список**: модель вправе запросить несколько вызовов за один ход;
- `arguments` — уже словарь: Ollama разобрала JSON за нас (в OpenAI-совместимых API пришла бы строка);
- сам запрос `query` модель **сочинила** — она могла переписать вопрос своими словами. Это фича (переформулировка!) и одновременно источник сюрпризов: за тем, *что* она ищет, интересно следить в трейсе.

Если вместо `tool_calls` пришёл обычный `content` — модель решила ответить без инструментов (для маленьких моделей это частый режим, когда вопрос кажется ей «простым»; system-промпт «пользуйся инструментами» с этим борется).

## 5. Цикл агента

Теперь — сердце урока, и оно помещается в тридцать строк. Правила цикла:

1. Позвали модель. Если в ответе **нет** `tool_calls` — это финальный ответ, выходим.
2. Если есть — выполняем **каждый** вызов, кладём результаты в диалог (`role: "tool"`), идём на следующую итерацию.
3. **`max_steps`** — обязателен: агент без лимита может зациклиться (искать одно и то же вечно). Это наш рубеж, модели он не подчиняется.
4. Ошибки инструмента и несуществующие инструменты **возвращаются модели текстом**, а не роняют цикл: агент умеет исправляться, труп цикла — нет.
5. Всё происходящее печатаем — **трейс** агента заменяет отладчик: без него вы не поймёте ни цену вопроса, ни причину странного ответа.

In [ ]:
SYSTEM_AGENT = (
    "Ты — ассистент по внутренней базе знаний компании «Векторика».\n"
    "У тебя есть инструменты: поиск по базе и калькулятор.\n"
    "Правила:\n"
    "1. Факты о компании бери ТОЛЬКО из результатов search_kb, не из памяти.\n"
    "2. Если поиск не нашёл ответа — переформулируй запрос и попробуй ещё раз "
    "(синонимы, официальные термины).\n"
    "3. Арифметику считай только через calc.\n"
    "4. Если ответа в базе нет и после переформулировок — честно скажи: "
    "«В базе знаний нет ответа на этот вопрос».\n"
    "5. Отвечай кратко, по-русски, в конце укажи документы-источники."
)


def run_agent(question: str, max_steps: int = 6, verbose: bool = True) -> dict:
    """Цикл агента: думай -> действуй -> наблюдай. Возвращает ответ + счётчики."""
    messages = [{"role": "system", "content": SYSTEM_AGENT},
                {"role": "user", "content": question}]
    llm_calls = 0
    tools_used = []

    for step in range(1, max_steps + 1):
        message = chat_with_tools(messages, TOOLS)
        llm_calls += 1
        messages.append(message)

        calls = message.get("tool_calls") or []
        if not calls:                              # инструменты не нужны - ответ
            return {"answer": message.get("content", ""), "steps": step,
                    "llm_calls": llm_calls, "tools_used": tools_used}

        for call in calls:                         # может быть НЕСКОЛЬКО вызовов
            name = call["function"]["name"]
            arguments = call["function"]["arguments"] or {}
            handler = TOOL_HANDLERS.get(name)
            if handler is None:                    # модель выдумала инструмент
                result = f"ошибка: инструмента {name} не существует"
            else:
                try:
                    result = handler(**arguments)
                except TypeError as error:         # модель прислала не те аргументы
                    result = f"ошибка аргументов: {error}"
            tools_used.append(name)
            if verbose:
                args_str = json.dumps(arguments, ensure_ascii=False)
                print(f"  шаг {step}: {name}({args_str})")
                print(f"      -> {str(result)[:90]}")
            messages.append({"role": "tool", "tool_name": name,
                             "content": str(result)})

    return {"answer": "Не удалось найти ответ за отведённое число шагов.",
            "steps": max_steps, "llm_calls": llm_calls, "tools_used": tools_used}


print("агент собран: max_steps ограничивает цикл, ошибки возвращаются модели текстом")

## 6. Три эксперимента

Погоняйте агента на трёх вопросах по нарастающей сложности. Для каждого — что *ожидается* и на что смотреть в трейсе (напоминание: у 3B-модели агентное поведение хрупкое, ваши трейсы могут отличаться — это часть эксперимента).

**Эксперимент 1 — простой вопрос.** Ожидание: один `search_kb`, затем ответ. Это агентный аналог конвейера — и он уже **дороже** конвейера (два вызова LLM вместо одного). Смотрите в трейсе, какой `query` модель сочинила.

In [ ]:
if OLLAMA_AVAILABLE and QDRANT_AVAILABLE:
    result = run_agent("Сколько дней отпуска положено в году?")
    print(f"\nответ: {result['answer']}")
    print(f"шагов: {result['steps']}, вызовов LLM: {result['llm_calls']}, "
          f"инструменты: {result['tools_used']}")
else:
    print("[пропущено: нужны Ollama и Qdrant]")

**Эксперимент 2 — вопрос с вычислением.** «Мне положено 28 дней отпуска, я уже взял 12 и хочу разбить остаток пополам — по сколько дней выйдет?» Ожидание: `search_kb` может и не понадобиться (числа в вопросе), но `calc("(28-12)/2")` — обязан появиться. Здесь видно разделение труда: модель *планирует* вычисление, считает — наш код. Заодно проверьте на своей машине подвох: одна из частей отпуска по правилам «Векторики» должна быть не короче 14 дней — заметит ли модель конфликт, если она нашла документ?

In [ ]:
if OLLAMA_AVAILABLE and QDRANT_AVAILABLE:
    result = run_agent("Мне положено 28 дней отпуска, я уже взял 12. "
                       "Хочу разбить остаток на две равные части - "
                       "по сколько дней получится?")
    print(f"\nответ: {result['answer']}")
    print(f"шагов: {result['steps']}, вызовов LLM: {result['llm_calls']}, "
          f"инструменты: {result['tools_used']}")
else:
    print("[пропущено: нужны Ollama и Qdrant]")

**Эксперимент 3 — канарейка.** «Упал прод, что делать в первую очередь?» — вопрос, который лексический конвейер проваливал во всех прогонах с урока 2.5 (синонимы!). У агента впервые есть **ход, которого не было у конвейера**: увидеть мусорную выдачу и поискать снова другими словами — подсказка «вместо „упал прод“ — „инцидент“, „дежурный“» лежит прямо в description инструмента.

Это самый интересный трейс урока. Возможные исходы:

- модель переформулирует запрос → найдёт `incidents` → ответит про war room и SEV1 — **агент победил там, где конвейер был обречён**;
- модель поищет один раз, получит мусор и честно откажется — дисциплина сработала, стратегия нет;
- модель ответит из весов, проигнорировав мусорную выдачу, — грубое нарушение grounding, и это аргумент, почему **рубежи из 3.4 нужны и агенту** (проверка цитат и self-check применимы к его финальному ответу точно так же).

Какой исход выпадет у 3B-модели — заранее не скажешь; на то и эксперимент.

In [ ]:
if OLLAMA_AVAILABLE and QDRANT_AVAILABLE:
    result = run_agent("Упал прод, что делать в первую очередь?")
    print(f"\nответ: {result['answer']}")
    print(f"шагов: {result['steps']}, вызовов LLM: {result['llm_calls']}, "
          f"инструменты: {result['tools_used']}")
    print("\nсмотрите в трейс: сколько раз агент искал и с какими формулировками?")
else:
    print("[пропущено: нужны Ollama и Qdrant]")

## 7. Дисциплина: безопасность, лимиты, бюджет

Агент — это модель, которой выдали руки. Прежде чем радоваться, перечитайте список обязанностей *нашего* кода — всё это уже встроено в `run_agent`, но заслуживает явного проговаривания:

1. **Аргументы инструментов — недоверенный ввод.** Модель галлюцинирует и в аргументах: не тот тип, лишние поля, код вместо арифметики. Валидация как для пользовательского ввода (наш `calc` — образец; для сложных инструментов — Pydantic-модель аргументов, урок 1.8).
2. **Инструменты этого урока — только читающие.** `search_kb` и `calc` не меняют мир. Пишущие инструменты (создать тикет, отправить письмо) — отдельная лига ответственности: подтверждение человеком, ограничение прав, журналирование. В модуле 7 вернёмся.
3. **`max_steps` — не подсказка, а стена.** Модель может зациклиться (одинаковый поиск снова и снова) — лимит шагов гарантирует остановку. Родственник `max_retries` из 3.4: модели нельзя отдавать контроль над числом попыток.
4. **Цена растёт линейно с шагами.** Каждый шаг = вызов LLM, и промпт с каждым шагом длиннее (в него въезжают все прошлые результаты поиска!). Счётчики `llm_calls`/`tools_used` в результате — зародыш бюджета и мониторинга (урок 3.7).
5. **Guardrails 3.4 никуда не делись.** Финальный ответ агента — такой же ответ: проверка цитат и self-check применимы к нему без изменений. Агентность меняет способ *добычи* контекста, а не правила его использования.

## ⚠️ Частые ошибки

| ошибка | чем оборачивается | лекарство |
|---|---|---|
| `eval()` в инструменте-калькуляторе | модель (или инъекция через документ!) выполняет произвольный код | разбор через `ast` с белым списком; аргументы = недоверенный ввод |
| цикл без `max_steps` | агент зацикливается, счёт за токены растёт | жёсткий лимит в коде цикла |
| исключение инструмента роняет цикл | один кривой аргумент — и диалог потерян | ошибки возвращать модели строкой: агент умеет исправляться |
| обработан только первый tool_call | модель просила два вызова, получила один результат — путается | `tool_calls` — список, выполнять все |
| расплывчатые description | модель выбирает не тот инструмент или не использует их вовсе | описание = промпт: когда использовать, что вернёт, что делать при неудаче |
| десяток инструментов у 3B-модели | путается в выборе тем сильнее, чем длиннее меню | маленьким моделям — 2–4 инструмента с контрастными описаниями |
| результат инструмента не добавлен в messages | модель не видит, что нашлось, и зовёт инструмент снова | каждый результат — сообщением `role: "tool"` |
| агент для задачи с фиксированным маршрутом | платите за LLM-решения там, где решает if | начинайте с конвейера; агент — для вариативных маршрутов |
| перенос кода с OpenAI-совместимого API как есть | там `arguments` — строка JSON, тут — уже словарь | проверить формат `arguments` своего API |

## 🏋️ Упражнения

**1. Инструмент `get_document`.** Поиск возвращает чанки, а иногда нужен весь документ (parent-document, урок 2.6). Напишите инструмент `get_document(doc_id)`, который возвращает полный текст документа по id (документы уже умеет загружать `load_documents` из `utils.py`), опишите его для модели — и придумайте вопрос, где агент должен сначала найти чанк, а потом запросить весь документ.

**2. Агент против конвейера — по метрикам.** У вас есть golden из урока 3.3 и агент. Напишите прогон агента по всем 13 вопросам со счётом: верные отказы на ловушках, вызовы LLM суммарно. Сравните с конвейером 3.4 по двум колонкам: качество отказов и цена. Прогноз: где агент выиграет, где проиграет?

**3. Бюджет вместо шагов.** `max_steps` ограничивает итерации, но шаги разные по цене: поиск дёшев, а каждый лишний вызов LLM с разросшимся контекстом — дорог. Перепишите цикл под бюджет: `max_llm_calls` и `max_tool_calls` отдельно, и при исчерпании — не обрыв, а финальный ход модели с инструкцией «инструменты закончились, ответь по тому, что уже нашлось».

## 📝 Мини-квиз

<details><summary><b>1. Что физически происходит, когда «модель вызывает функцию»?</b></summary>

Модель генерирует JSON с именем инструмента и аргументами — это текст, такой же, как любой её ответ. Вызывает функцию наш код: читает JSON, валидирует, исполняет настоящую функцию и кладёт результат в диалог сообщением `role: "tool"`. У модели нет ни доступа к коду, ни сети — вся власть (и вся ответственность) у нашего цикла.
</details>

<details><summary><b>2. Чем агент отличается от конвейера 3.1–3.4 и когда конвейер лучше?</b></summary>

В конвейере маршрут фиксирован кодом (поиск → промпт → генерация), у агента маршрут выбирает модель на каждом шаге. Конвейер лучше, когда маршрут известен заранее: он дешевле (1–2 вызова LLM против вызова на каждый шаг), быстрее и предсказуемее — его проще тестировать и мерить. Агент оправдан, когда маршрут вариативен: переформулировать поиск, выбрать источник, посчитать.
</details>

<details><summary><b>3. Зачем max_steps, если модель обычно отвечает за 1–2 шага?</b></summary>

Затем же, зачем max_retries в 3.4: «обычно» — не гарантия. Агент может зациклиться (искать одно и то же, вызывать инструмент с теми же аргументами), и каждый лишний круг — деньги и латентность. Лимит живёт в нашем коде и модели не подчиняется — это стена, а не просьба.
</details>

<details><summary><b>4. Почему аргументы tool_call нужно валидировать, они же от нашей модели?</b></summary>

Модель галлюцинирует и в аргументах: не тот тип, выдуманные поля, а в худшем случае — код вместо данных (наш пример: `__import__('os')...` в калькуляторе). Плюс аргументы могут быть отравлены содержимым документов (prompt injection через базу знаний — модуль 4). Правило: аргументы инструмента = недоверенный ввод, валидация как у формы на сайте.
</details>

<details><summary><b>5. Инструмент упал с исключением. Что должен сделать цикл и почему?</b></summary>

Поймать исключение и вернуть модели текст ошибки сообщением `role: "tool"`. Агент, увидев «ошибка аргументов: ...», может исправиться на следующем шаге — переформулировать вызов или выбрать другой инструмент. Упавший цикл не может ничего, а пользователь получает стектрейс вместо ответа.
</details>

<details><summary><b>6. Почему description инструмента называют промптом?</b></summary>

Потому что модель принимает решение «какой инструмент и когда» исключительно по name и description — это текст, который она читает так же, как system-промпт. Наш search_kb несёт в description целую стратегию: когда использовать, и что при неудаче переформулировать запрос официальными терминами. Меняешь описание — меняется поведение агента, без единой строчки кода.
</details>

## Решения

**Решение 1 — инструмент `get_document`.**

In [ ]:
from utils import load_documents

DOCS_BY_ID = {doc.doc_id: doc for doc in load_documents("data")}


def tool_get_document(doc_id: str) -> str:
    doc = DOCS_BY_ID.get(str(doc_id).strip())
    if doc is None:
        known = ", ".join(sorted(DOCS_BY_ID))
        return f"ошибка: документа {doc_id!r} нет. Есть: {known}"
    return f"[{doc.doc_id}] {doc.title}\n\n{doc.text}"


TOOLS_V2 = TOOLS + [
    {"type": "function", "function": {
        "name": "get_document",
        "description": ("Полный текст документа базы знаний по его id "
                        "(id видно в результатах search_kb в квадратных "
                        "скобках). Используй, когда найденного фрагмента "
                        "мало и нужен весь документ целиком."),
        "parameters": {"type": "object",
                       "properties": {"doc_id": {
                           "type": "string",
                           "description": "id документа, например otpusk"}},
                       "required": ["doc_id"]},
    }},
]
TOOL_HANDLERS["get_document"] = tool_get_document

# механика проверяется без LLM:
print(tool_get_document("otpusk")[:100], "...")
print(tool_get_document("nesuschestvuyuschiy")[:100])

# вопрос, где чанка мало, а нужен весь документ (проверьте с моделью):
# run_agent("Перечисли ВСЕ правила, связанные с отпуском, по пунктам")
# ожидание: search_kb найдёт чанк -> модель увидит id -> get_document("otpusk")

Обратите внимание на текст ошибки в `get_document`: он **перечисляет валидные id**. Ошибка инструмента — тоже интерфейс: чем информативнее текст, тем выше шанс, что агент исправится со второй попытки (тот же принцип, что подсказка тем в отказе из 3.4).

**Решение 2 — агент против конвейера по метрикам.** Каркас прогона; сравнение заполните числами со своей машины.

In [ ]:
if OLLAMA_AVAILABLE and QDRANT_AVAILABLE:
    golden = json.loads(Path("eval/golden.json").read_text(encoding="utf-8"))

    REFUSAL_PATTERNS = ("нет ответа", "нет информации", "не нашлось",
                        "не могу ответить", "нет данных")

    def is_refusal(answer: str) -> bool:
        return any(p in answer.lower() for p in REFUSAL_PATTERNS)

    total_llm = true_refusals = false_refusals = 0
    for item in golden:
        result = run_agent(item["question"], verbose=False)
        total_llm += result["llm_calls"]
        refused = is_refusal(result["answer"])
        if item["answerable"] and refused:
            false_refusals += 1
        if not item["answerable"] and refused:
            true_refusals += 1
        mark = "отказ" if refused else "ответ"
        print(f"  [{mark}] {item['id']:<22} шагов={result['steps']} "
              f"llm={result['llm_calls']}")

    n_traps = sum(not g["answerable"] for g in golden)
    print(f"\nверные отказы:  {true_refusals}/{n_traps}")
    print(f"ложные отказы:  {false_refusals}/{len(golden) - n_traps}")
    print(f"вызовов LLM:    {total_llm} "
          f"(конвейер 3.1 сделал бы {len(golden)}, guarded-конвейер 3.4 - ~2x)")
    # Прогноз для сравнения: агент имеет шанс выиграть на канарейке
    # (переформулировка!) и обязан проиграть по цене - каждый вопрос
    # стоит 2+ вызова, а длина промпта растёт с каждым шагом.
else:
    print("[пропущено: нужны Ollama и Qdrant]")

**Решение 3 — бюджет вместо шагов.** Ключевая деталь — «последний ход»: при исчерпании бюджета мы не обрываем диалог, а делаем финальный вызов **без инструментов** и с инструкцией ответить по уже собранному. Агент завершает работу штатно, а не падает с «лимит исчерпан».

In [ ]:
def run_agent_budget(question: str, max_llm_calls: int = 5,
                     max_tool_calls: int = 4, verbose: bool = True) -> dict:
    """Цикл агента с раздельным бюджетом на вызовы LLM и инструментов."""
    messages = [{"role": "system", "content": SYSTEM_AGENT},
                {"role": "user", "content": question}]
    llm_calls = tool_calls = 0

    while llm_calls < max_llm_calls - 1:        # -1: резервируем последний ход
        message = chat_with_tools(messages, TOOLS)
        llm_calls += 1
        messages.append(message)

        calls = message.get("tool_calls") or []
        if not calls:
            return {"answer": message.get("content", ""),
                    "llm_calls": llm_calls, "tool_calls": tool_calls}

        for call in calls:
            if tool_calls >= max_tool_calls:    # бюджет инструментов кончился
                messages.append({"role": "tool",
                                 "tool_name": call["function"]["name"],
                                 "content": "бюджет инструментов исчерпан"})
                continue
            name = call["function"]["name"]
            arguments = call["function"]["arguments"] or {}
            handler = TOOL_HANDLERS.get(name)
            try:
                result = (handler(**arguments) if handler
                          else f"ошибка: инструмента {name} нет")
            except TypeError as error:
                result = f"ошибка аргументов: {error}"
            tool_calls += 1
            if verbose:
                print(f"  {name}({json.dumps(arguments, ensure_ascii=False)}) "
                      f"[бюджет: llm {llm_calls}/{max_llm_calls}, "
                      f"tools {tool_calls}/{max_tool_calls}]")
            messages.append({"role": "tool", "tool_name": name,
                             "content": str(result)})

    # последний ход: инструментов больше нет, отвечай по собранному
    messages.append({"role": "user", "content":
                     "Инструменты закончились. Дай финальный ответ по тому, "
                     "что уже нашлось. Если данных не хватает — скажи об этом."})
    final = chat_with_tools(messages, tools=[])
    return {"answer": final.get("content", ""),
            "llm_calls": llm_calls + 1, "tool_calls": tool_calls}


print("бюджетный агент собран; сравните с run_agent на канарейке:")
print("run_agent_budget('Упал прод, что делать в первую очередь?')")

In [ ]:
run_agent_budget('Упал прод, что делать в первую очередь?')

## Итоги

- **Function calling — это генерация JSON, а не магия.** Модель пишет намерение, исполняет — наш код; поэтому аргументы инструментов — недоверенный ввод, а `eval()` в инструменте — дыра (наш `calc` разбирает выражения через `ast` с белым списком).
- **Цикл агента — 30 строк**: думай → действуй → наблюдай, `max_steps` стеной, ошибки — модели текстом (она умеет исправляться), все `tool_calls` из списка — выполняются, каждый результат — в диалог ролью `tool`.
- **Описание инструмента — это промпт.** Стратегия «не нашлось — переформулируй официальными терминами» живёт в description `search_kb` и меняет поведение агента без кода. Именно она даёт канарейке «Упал прод» первый реальный шанс с урока 2.5.
- **Агент дороже конвейера по построению**: вызов LLM на каждый шаг, растущий промпт. Начинайте с конвейера; агент — для маршрутов, которые не прибить гвоздями.
- **Guardrails 3.4 применимы и к агенту**: его финальный ответ проверяется цитатами и self-check ровно так же. Агентность меняет добычу контекста, не правила его использования.

| приём урока | родом из |
|---|---|
| ретривер как инструмент, скоры в выдаче | уроки 2.4, 3.2 |
| JSON Schema аргументов | Pydantic, урок 1.8 |
| grounding-правила в system-промпте агента | уроки 1.2, 3.1, 3.3 |
| лимиты (max_steps, бюджет), ошибки как текст | max_retries и отказ-как-фича из 3.4 |
| канарейка «Упал прод» как полигон | уроки 2.5, 3.3 |

**Что дальше.** Наш агент живёт внутри одной функции: история — локальная переменная, упал процесс — пропал диалог, а маршрут «думай-действуй» нигде не описан, кроме как в цикле. Урок 3.6 — **LangGraph**: состояние агента как явный граф с узлами и рёбрами, checkpointer для памяти диалога между запусками, и честный разбор — что фреймворк даёт по сравнению с нашими 30 строками и чего стоит.